# 0002 - Add Two Numbers

Neste notebook, vamos resolver o problema **Add Two Numbers** de um jeito didático, começando por uma abordagem mais ingênua, passando por uma melhoria intermediária e terminando na solução ideal.

A ideia é que este conteúdo possa ser estudado com calma, entendido sem depender do enunciado original e, no futuro, exportado para um **Jupyter Book** com boa apresentação no GitHub Pages.


## Enunciado

### Enunciado original

Enunciado original

### Enunciado traduzido em linguagem simples

Cada lista encadeada guarda um número, mas os dígitos aparecem ao contrário.
Então `2 -> 4 -> 3` representa o número `342`, e `5 -> 6 -> 4` representa `465`.
O desafio é somar esses dois números e devolver uma nova lista encadeada com o resultado também em ordem reversa.


## Imports

Colocar aqui todos os imports necessários


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')


@dataclass
class ListNode:
    """Representa um nó simples de uma lista encadeada.

    A estrutura segue o formato clássico usado em problemas de LeetCode:
    cada nó guarda um valor inteiro e uma referência para o próximo nó.

    Parâmetros:
    ----------
    val : int, default = 0
        Valor armazenado no nó.
    next : ListNode | None, default = None
        Referência para o próximo nó da lista.
    """

    val: int = 0
    next: ListNode | None = None


def list_to_linked_list(values: list[int]) -> ListNode | None:
    """Converte uma lista Python em uma lista encadeada.

    Parâmetros:
    ----------
    values : list[int]
        Lista de valores que será transformada em linked list.

    Retorno:
    -------
    ListNode | None
        Cabeça da lista encadeada. Retorna `None` se a lista de entrada estiver vazia.

    Exemplos de uso:
    ----------------
    >>> list_to_linked_list([2, 4, 3])
    ListNode(val=2, next=ListNode(val=4, next=ListNode(val=3, next=None)))
    """
    if not values:
        return None

    head = ListNode(values[0])
    current = head

    for value in values[1:]:
        current.next = ListNode(value)
        current = current.next

    return head


def linked_list_to_list(head: ListNode | None) -> list[int]:
    """Converte uma lista encadeada em uma lista Python.

    Parâmetros:
    ----------
    head : ListNode | None
        Cabeça da lista encadeada.

    Retorno:
    -------
    list[int]
        Valores da linked list na mesma ordem em que aparecem.
    """
    valores: list[int] = []
    atual = head

    while atual is not None:
        valores.append(atual.val)
        atual = atual.next

    return valores


def print_linked_list(head: ListNode | None) -> str:
    """Retorna uma representação amigável da lista encadeada.

    Parâmetros:
    ----------
    head : ListNode | None
        Cabeça da lista encadeada.

    Retorno:
    -------
    str
        Texto com os nós separados por setas.
    """
    valores = [str(valor) for valor in linked_list_to_list(head)]
    return ' -> '.join(valores) if valores else 'lista vazia'


def linked_list_to_int(head: ListNode | None) -> int:
    """Converte uma lista encadeada em inteiro considerando dígitos reversos.

    Parâmetros:
    ----------
    head : ListNode | None
        Cabeça da lista encadeada com os dígitos em ordem reversa.

    Retorno:
    -------
    int
        Número inteiro correspondente à lista.

    Exemplos de uso:
    ----------------
    >>> linked_list_to_int(list_to_linked_list([2, 4, 3]))
    342
    """
    multiplo = 1
    resultado = 0
    atual = head

    while atual is not None:
        resultado += atual.val * multiplo
        multiplo *= 10
        atual = atual.next

    return resultado


def int_to_linked_list(number: int) -> ListNode:
    """Converte um inteiro não negativo em lista encadeada reversa.

    Parâmetros:
    ----------
    number : int
        Número inteiro não negativo que será convertido.

    Retorno:
    -------
    ListNode
        Lista encadeada contendo os dígitos em ordem reversa.

    Exceções:
    --------
    ValueError
        Levantada se `number` for negativo.

    Exemplos de uso:
    ----------------
    >>> print_linked_list(int_to_linked_list(807))
    '7 -> 0 -> 8'
    """
    if number < 0:
        raise ValueError('O número deve ser não negativo.')

    if number == 0:
        return ListNode(0)

    head: ListNode | None = None
    tail: ListNode | None = None

    while number > 0:
        number, digit = divmod(number, 10)
        node = ListNode(digit)

        if head is None:
            head = node
            tail = node
        else:
            assert tail is not None
            tail.next = node
            tail = node

    assert head is not None
    return head


def mostrar_exemplo(l1_values: list[int], l2_values: list[int]) -> None:
    """Exibe uma leitura textual do exemplo com listas e números."""
    l1 = list_to_linked_list(l1_values)
    l2 = list_to_linked_list(l2_values)
    print(f"l1 como linked list: {print_linked_list(l1)}")
    print(f"l2 como linked list: {print_linked_list(l2)}")
    print(f"l1 como número: {linked_list_to_int(l1)}")
    print(f"l2 como número: {linked_list_to_int(l2)}")
    print(f"soma esperada: {linked_list_to_int(l1) + linked_list_to_int(l2)}")


def linked_list_equals(head: ListNode | None, expected: list[int]) -> bool:
    """Compara uma linked list com uma lista Python esperada."""
    return linked_list_to_list(head) == expected


def benchmark_solution(func, cases: list[tuple[list[int], list[int]]], repetitions: int = 20) -> float:
    """Mede o tempo médio de uma solução em um conjunto de casos.

    Parâmetros:
    ----------
    func : callable
        Função que resolve o problema.
    cases : list[tuple[list[int], list[int]]]
        Casos de entrada no formato de listas de dígitos reversos.
    repetitions : int, default = 20
        Número de repetições para média do tempo.

    Retorno:
    -------
    float
        Tempo médio em segundos.
    """
    tempos: list[float] = []

    for _ in range(repetitions):
        inicio = perf_counter()
        for l1_values, l2_values in cases:
            l1 = list_to_linked_list(l1_values)
            l2 = list_to_linked_list(l2_values)
            resultado = func(l1, l2)
            assert isinstance(resultado, ListNode)
        fim = perf_counter()
        tempos.append(fim - inicio)

    return float(np.mean(tempos))


print('Estruturas auxiliares prontas.')


## Funções uteis

**Imports**

Colocar aqui todos os imports necessários

**Funções uteis**

**Intuição geral**

Antes de escrever código, vale alinhar a intuição.

**O que é uma linked list?**

Uma lista encadeada é uma estrutura formada por nós. Cada nó guarda um valor e uma referência para o próximo nó.
Diferente de uma lista Python, os elementos não ficam necessariamente em posições contíguas de memória.

**O que significa ordem reversa?**

Se os dígitos estão em ordem reversa, o primeiro nó da lista é o dígito das unidades, o segundo é o das dezenas, o terceiro é o das centenas, e assim por diante.

Por exemplo:

- `2 -> 4 -> 3` representa `342`
- `5 -> 6 -> 4` representa `465`
- a soma é `807`, então o retorno é `7 -> 0 -> 8`

**Ideia central do problema**

A parte mais importante é perceber que a soma precisa respeitar o **vai-um** (`carry`).
Quando a soma de dois dígitos passa de `9`, carregamos `1` para a próxima posição.

É exatamente isso que a solução ótima vai fazer, dígito por dígito.

**Preparação do ambiente**

Antes das soluções, vamos criar uma estrutura de `ListNode` e algumas funções auxiliares para facilitar testes, explicações e visualização.

Essas funções tornam o notebook mais didático e deixam as soluções principais mais limpas.

Vamos olhar para o primeiro exemplo de forma mais concreta.

- `2 -> 4 -> 3` representa `342`
- `5 -> 6 -> 4` representa `465`
- `342 + 465 = 807`
- o resultado volta como `7 -> 0 -> 8`

Esse tipo de leitura ajuda muito a entender por que o problema pede uma lista reversa.


In [ ]:
mostrar_exemplo([2, 4, 3], [5, 6, 4])


## Solução 1 - Força bruta


### Descrição da solução

**Descrição da solução**

A primeira solução é a mais ingênua possível: transformar as duas listas encadeadas em números inteiros, somar os números e depois converter o resultado de volta para uma lista encadeada.

Essa abordagem funciona porque ela traduz o problema para algo muito familiar: soma de inteiros.
Mas ela também revela uma limitação importante: precisamos materializar o número inteiro completo antes de fazer a soma.

**Por que isso é uma solução bruta?**

Porque ela resolve o problema usando uma transformação completa das estruturas, sem explorar diretamente a natureza dígito a dígito da lista encadeada.
Ela é simples e fácil de pensar, mas não é a forma mais elegante nem a mais eficiente em termos de estrutura.

**Como chegamos ao tempo `O(m + n)`?**

Se `m` é o tamanho da primeira lista e `n` é o da segunda, precisamos percorrer cada uma delas para reconstruir os números.
Depois fazemos a soma e convertemos o resultado de volta para lista.

Em termos de ideia, o custo total cresce linearmente com o número de nós processados.
Mesmo que a operação de somar inteiros pareça "instantânea" para valores pequenos, o ponto principal aqui é que a estratégia inteira depende de percorrer as listas e reconstruir estruturas completas.

**Por que o espaço é maior?**

Porque essa solução materializa o número inteiro completo e depois materializa a lista resultado. Ela é didática, mas não é a forma mais econômica de trabalhar com listas encadeadas.

**Limitação importante**

Essa abordagem é simples, mas faz uma conversão completa antes de produzir o resultado.
Ela funciona muito bem para entender o problema, mas não é a melhor escolha quando queremos pensar de forma mais direta sobre os nós da lista.

**Implementação da força bruta**

Vamos validar a solução mais ingênua com os exemplos do enunciado e com alguns casos extras para garantir que o comportamento está correto.


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(m + n)`
- **Espaço extra:** `O(m + n)` se considerarmos a criação de estruturas auxiliares para a conversão e para o resultado


### Mini walkthrough

**Intuição**

- caminhar pelas listas e reconstruir os dois números;
- somar os inteiros;
- converter o resultado em uma nova lista encadeada.

**Passo a passo com o exemplo**

Para `l1 = 2 -> 4 -> 3` e `l2 = 5 -> 6 -> 4`:

- `l1` vira `342`
- `l2` vira `465`
- `342 + 465 = 807`
- `807` vira `7 -> 0 -> 8`


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da força bruta


In [ ]:
def add_two_numbers_brute_force(l1: ListNode | None, l2: ListNode | None) -> ListNode:
    """Soma duas listas encadeadas convertendo tudo para inteiros.

    Descrição:
    ----------
    Esta é a solução mais ingênua do notebook. Primeiro reconstruímos os dois
    números inteiros completos a partir das listas encadeadas, depois somamos
    os valores e, por fim, transformamos o resultado novamente em uma lista.

    Parâmetros:
    ----------
    l1 : ListNode | None
        Cabeça da primeira lista encadeada.
    l2 : ListNode | None
        Cabeça da segunda lista encadeada.

    Retorno:
    -------
    ListNode
        Lista encadeada com a soma em ordem reversa.

    Exceções:
    --------
    ValueError
        Levantada se alguma lista for vazia, já que o problema assume listas não vazias.

    Exemplos de uso:
    ----------------
    >>> resultado = add_two_numbers_brute_force(list_to_linked_list([2, 4, 3]), list_to_linked_list([5, 6, 4]))
    >>> linked_list_to_list(resultado)
    [7, 0, 8]
    """
    if l1 is None or l2 is None:
        raise ValueError('As duas listas devem ser não vazias.')

    numero_1 = linked_list_to_int(l1)
    numero_2 = linked_list_to_int(l2)
    soma = numero_1 + numero_2
    return int_to_linked_list(soma)


### Testes


In [ ]:
casos_teste = [
    ([2, 4, 3], [5, 6, 4], [7, 0, 8]),
    ([0], [0], [0]),
    ([9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9], [8, 9, 9, 9, 0, 0, 0, 1]),
    ([1], [9, 9, 9], [0, 0, 0, 1]),
    ([9, 9, 9], [1], [0, 0, 0, 1]),
]

for l1_values, l2_values, esperado in casos_teste:
    l1 = list_to_linked_list(l1_values)
    l2 = list_to_linked_list(l2_values)
    resultado = add_two_numbers_brute_force(l1, l2)
    assert linked_list_equals(resultado, esperado), (
        f'Falha em l1={l1_values}, l2={l2_values}: obtido {linked_list_to_list(resultado)}, esperado {esperado}'
    )

print('Todos os testes da Solução 1 passaram.')


## Solução 2 - Melhorias da solução 1


### Descrição da solução

**Descrição da solução**

A segunda solução melhora a ideia anterior, mas ainda não é a solução ideal.

Aqui, vamos continuar trabalhando com uma abordagem em que fazemos uma conversão intermediária das listas, mas sem depender da criação de um número inteiro grande como etapa central.
Em vez disso, vamos somar diretamente sobre os dígitos em uma estrutura auxiliar simples.

**Por que isso é uma melhoria?**

- evita depender de uma conversão completa para inteiro;
- trabalha diretamente com os dígitos;
- continua sendo fácil de entender;
- mantém a solução linear.

Essa versão ainda não é a mais elegante possível, porque continua materializando as listas em memória antes da soma final.
Mas ela já se aproxima melhor do raciocínio do problema do que a solução 1.

**Como chegamos ao tempo `O(m + n)`?**

Precisamos percorrer cada lista para convertê-la em uma lista Python de dígitos.
Depois, percorremos os dígitos uma única vez para fazer a soma com carry.

Mesmo havendo múltiplas etapas, cada nó participa de um número constante de operações.
Então o crescimento continua linear.

**Por que essa solução é melhor que a anterior?**

Porque ela deixa de depender de um número inteiro gigante como representação intermediária.
Isso torna a implementação mais próxima da estrutura real do problema e reduz uma camada de conversão conceitual.

**O que ainda impede essa solução de ser a ideal?**

Ela ainda precisa transformar as listas encadeadas em estruturas auxiliares antes da soma.
Ou seja, ainda não estamos processando os nós de maneira totalmente incremental.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(m + n)`
- **Espaço extra:** `O(m + n)`


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def add_two_numbers_better(l1: ListNode | None, l2: ListNode | None) -> ListNode:
    """Soma duas listas encadeadas usando listas auxiliares de dígitos.

    Descrição:
    ----------
    Esta solução ainda faz uma etapa intermediária de materialização, mas evita
    transformar os dígitos em um único inteiro grande. Em vez disso, convertemos
    as listas encadeadas para listas Python de dígitos e fazemos a soma com carry
    de forma explícita.

    Parâmetros:
    ----------
    l1 : ListNode | None
        Cabeça da primeira lista encadeada.
    l2 : ListNode | None
        Cabeça da segunda lista encadeada.

    Retorno:
    -------
    ListNode
        Lista encadeada com a soma em ordem reversa.

    Exceções:
    --------
    ValueError
        Levantada se alguma lista for vazia.

    Exemplos de uso:
    ----------------
    >>> resultado = add_two_numbers_better(list_to_linked_list([2, 4, 3]), list_to_linked_list([5, 6, 4]))
    >>> linked_list_to_list(resultado)
    [7, 0, 8]
    """
    if l1 is None or l2 is None:
        raise ValueError('As duas listas devem ser não vazias.')

    digitos_1 = linked_list_to_list(l1)
    digitos_2 = linked_list_to_list(l2)
    resultado: list[int] = []
    carry = 0
    limite = max(len(digitos_1), len(digitos_2))

    for indice in range(limite):
        valor_1 = digitos_1[indice] if indice < len(digitos_1) else 0
        valor_2 = digitos_2[indice] if indice < len(digitos_2) else 0
        soma = valor_1 + valor_2 + carry
        resultado.append(soma % 10)
        carry = soma // 10

    if carry:
        resultado.append(carry)

    return list_to_linked_list(resultado)


### Testes


Agora vamos garantir que a solução intermediária também funciona nos mesmos cenários de teste.


In [ ]:
for l1_values, l2_values, esperado in casos_teste:
    l1 = list_to_linked_list(l1_values)
    l2 = list_to_linked_list(l2_values)
    resultado = add_two_numbers_better(l1, l2)
    assert linked_list_equals(resultado, esperado), (
        f'Falha em l1={l1_values}, l2={l2_values}: obtido {linked_list_to_list(resultado)}, esperado {esperado}'
    )

print('Todos os testes da Solução 2 passaram.')


## Solução 3 - Melhor solução


### Descrição da solução

**Descrição da solução**

Agora chegamos à solução ideal.

A melhor maneira de resolver este problema é percorrer as duas listas simultaneamente, somando um dígito de cada vez e controlando o `carry`.
Assim, não precisamos transformar as listas em números inteiros nem criar estruturas intermediárias maiores do que o necessário.
Como os dígitos já estão em ordem reversa, a leitura da lista já nos entrega exatamente a ordem em que a soma precisa acontecer.

**O que é o `carry`?**

O `carry` é o famoso "vai um".
Quando a soma de dois dígitos passa de `9`, a parte das unidades fica no nó atual e a dezena é levada para a próxima posição.
Por exemplo, em `4 + 6 = 10`, escrevemos `0` e carregamos `1` para a próxima soma.

**Observação sobre a checagem de listas vazias**

O enunciado garante que as listas são não vazias.
Por isso, a versão principal da solução não precisa validar `None` como caso de erro.
Essa checagem só seria necessária em um código mais defensivo fora do contrato do problema.

**Implementação da melhoria**


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

**Por que essa é a solução ótima?**

- ela usa diretamente a estrutura do problema;
- faz uma única passagem pelas listas;
- não depende de conversões desnecessárias;
- lida com o `carry` de forma natural;
- alcança a melhor complexidade assintótica possível para esse cenário.

No contexto do LeetCode, o ranking de runtime e memory pode oscilar entre submissões.
Isso significa que esse tipo de métrica não deve ser usado como verdade absoluta sobre a qualidade algorítmica.
Ela pode variar com o ambiente, com o tamanho da entrada e até com ruídos do próprio sistema de avaliação.

Depois que chegamos a `O(max(m, n))`, o que sobra são apenas micro-otimizações de implementação.
Isso pode mudar um pouco a constante de tempo, mas não muda a classe de complexidade.

**Comparação entre a versão didática e a versão enxuta**

- a versão didática é mais explícita e melhor para estudo;
- a versão enxuta usa `divmod` e fica mais compacta;
- ambas têm a mesma complexidade;
- a diferença principal é legibilidade versus concisão.

Para aprender, a versão didática costuma ser a mais confortável.
Para submissão, a versão enxuta costuma ficar mais agradável.


### Mini walkthrough

**Intuição**

- começamos no primeiro nó de cada lista;
- somamos os valores atuais e o `carry`;
- o dígito da unidade entra no resultado;
- o dígito da dezena vira o novo `carry`;
- avançamos para o próximo nó;
- continuamos até terminar as listas e o `carry`.

Para `2 -> 4 -> 3` e `5 -> 6 -> 4`:

- primeira posição: `2 + 5 = 7`, carry `0`
- segunda posição: `4 + 6 = 10`, escrevemos `0`, carry `1`
- terceira posição: `3 + 4 + 1 = 8`, carry `0`
- resultado final: `7 -> 0 -> 8`


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def add_two_numbers_optimized_didatica(l1: ListNode, l2: ListNode) -> ListNode:
    """Soma duas listas encadeadas dígito a dígito, de forma explícita e didática.

    Descrição:
    ----------
    Esta é a versão mais explicativa da solução ótima. Percorremos as duas
    listas ao mesmo tempo, somando os dígitos atuais e o `carry`. A cada passo,
    construímos um novo nó com o dígito resultante e carregamos o vai-um para a
    próxima posição.

    Parâmetros:
    ----------
    l1 : ListNode
        Cabeça da primeira lista encadeada.
    l2 : ListNode
        Cabeça da segunda lista encadeada.

    Retorno:
    -------
    ListNode
        Cabeça da lista encadeada contendo a soma em ordem reversa.

    Exceções:
    --------
    ValueError
        Levantada se alguma referência for inválida em um uso fora do contrato do problema.

    Exemplos de uso:
    ----------------
    >>> resultado = add_two_numbers_optimized_didatica(list_to_linked_list([2, 4, 3]), list_to_linked_list([5, 6, 4]))
    >>> linked_list_to_list(resultado)
    [7, 0, 8]
    """
    dummy = ListNode(0)
    atual = dummy
    carry = 0
    p1 = l1
    p2 = l2

    while p1 is not None or p2 is not None or carry:
        valor_1 = p1.val if p1 is not None else 0
        valor_2 = p2.val if p2 is not None else 0
        soma = valor_1 + valor_2 + carry
        carry = soma // 10
        digito = soma % 10
        atual.next = ListNode(digito)
        atual = atual.next

        if p1 is not None:
            p1 = p1.next
        if p2 is not None:
            p2 = p2.next

    return dummy.next


### Testes


Exibição de alguns resultados da execução da solução.


## Solução 4 - Solução enxuta para submissão no leetcode


### Descrição da solução

**Descrição da solução**

A implementação abaixo é a mesma solução ótima, mas em uma forma mais compacta.
Ela usa `divmod` para obter, de uma vez só, o novo `carry` e o dígito a ser armazenado no nó atual.

Essa versão é boa para submissão e revisão rápida de código, enquanto a versão didática é melhor para estudo.

**Como chegamos ao tempo `O(max(m, n))`?**

Percorremos as duas listas ao mesmo tempo até que ambas acabem.
Cada iteração faz um número constante de operações.

Se uma lista tem `m` nós e a outra `n`, o loop principal executa no máximo `max(m, n)` vezes, mais uma possível iteração final por causa do `carry`.
Isso continua sendo linear.

**Por que essa é a solução preferida em entrevistas e produção?**

Porque ela resolve o problema diretamente, sem conversões intermediárias desnecessárias.
Além disso, deixa claro que entendemos a relação entre a representação reversa e a soma com carry.

**O que ela ensina?**

Esse problema é uma boa lição sobre como aproveitar a estrutura dos dados em vez de lutar contra ela.
Quando a lista já está em ordem reversa, a soma dígito a dígito vira naturalmente a melhor estratégia.

**Implementação da melhoria**

Vamos testar as duas variantes da solução ótima nos mesmos cenários, incluindo exemplos do enunciado, listas de tamanhos diferentes, carry no final, sequência com muitos noves e o caso mínimo `[0] + [0]`.


### Ordem de complexidade (memória e processamento)

**Ordem de complexidade (memória e processamento)**

- **Tempo:** `O(max(m, n))`
- **Espaço extra:** `O(max(m, n))` para a lista de saída

**Comparação prática**

As duas versões da solução ótima têm a mesma complexidade assintótica.
A diferença está em como o código é apresentado:

- a versão didática é mais explícita e excelente para estudo;
- a versão enxuta é mais compacta e costuma ser agradável para submissão;
- ambas seguem a mesma estratégia e produzem o mesmo resultado.

Em termos práticos, a versão enxuta pode parecer um pouco melhor por usar `divmod` e menos linhas.
Mesmo assim, o ganho é de constante, não de ordem de crescimento.


### Mini walkthrough

Exibição de alguns exemplos de como o algoritmo funciona.


### Exemplo de execuçaõ em ASCII sketch

**Exemplo de execuçaõ em ASCII sketch**

Exibição de algum exemplo do passo a passo do algoritmo.


### Implementação da melhoria


In [ ]:
def add_two_numbers_optimized(l1: ListNode, l2: ListNode) -> ListNode:
    # Versão enxuta para submissão.
    dummy = ListNode(0)
    atual = dummy
    carry = 0

    while l1 is not None or l2 is not None or carry:
        valor_1 = l1.val if l1 is not None else 0
        valor_2 = l2.val if l2 is not None else 0
        carry, digito = divmod(valor_1 + valor_2 + carry, 10)
        atual.next = ListNode(digito)
        atual = atual.next

        if l1 is not None:
            l1 = l1.next
        if l2 is not None:
            l2 = l2.next

    return dummy.next


### Testes


In [ ]:
casos_otimos = [
    ([2, 4, 3], [5, 6, 4], [7, 0, 8]),
    ([0], [0], [0]),
    ([9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9], [8, 9, 9, 9, 0, 0, 0, 1]),
    ([1], [9, 9, 9], [0, 0, 0, 1]),
    ([9, 9, 9], [1], [0, 0, 0, 1]),
    ([5, 6, 4], [2, 4, 3], [7, 0, 8]),
    ([9, 9, 9, 9], [1], [0, 0, 0, 0, 1]),
]

for l1_values, l2_values, esperado in casos_otimos:
    l1 = list_to_linked_list(l1_values)
    l2 = list_to_linked_list(l2_values)
    resultado_didatico = add_two_numbers_optimized_didatica(l1, l2)
    assert linked_list_equals(resultado_didatico, esperado), (
        f'Falha didática em l1={l1_values}, l2={l2_values}: obtido {linked_list_to_list(resultado_didatico)}, esperado {esperado}'
    )

for l1_values, l2_values, esperado in casos_otimos:
    l1 = list_to_linked_list(l1_values)
    l2 = list_to_linked_list(l2_values)
    resultado_enxuto = add_two_numbers_optimized(l1, l2)
    assert linked_list_equals(resultado_enxuto, esperado), (
        f'Falha enxuta em l1={l1_values}, l2={l2_values}: obtido {linked_list_to_list(resultado_enxuto)}, esperado {esperado}'
    )

print('Todos os testes das duas variantes da Solução 3 passaram.')


## Benchmark

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Construção de casos de testes para progressivos para mostrar
que as soluções se tornam cada vez mais otimizadas.
Exibir gráficos e tabelas de comparação entre as soluções pertinentes.

Neste problema, quase todas as soluções são lineares, então a diferença mais importante está nas **constantes** e na quantidade de conversões intermediárias.

Por isso, em vez de tentar inventar um benchmark muito rigoroso, vamos fazer uma visualização didática comparando:

- a solução 1, que reconstrói os números inteiros completos;
- a solução 2, que trabalha com listas de dígitos;
- a solução 3, que processa os nós diretamente.

A ideia não é provar formalmente a complexidade, e sim enxergar como os custos tendem a crescer conforme o tamanho da entrada aumenta.

**Metodologia**

Agora vamos medir o tempo médio das três soluções em casos crescentes.
Esse teste é apenas para visualização da tendência prática. Ele não prova formalmente a complexidade, mas ajuda a observar que a solução ótima costuma ser a mais direta e simples.

Como o problema tem limite pequeno de nós, um benchmark curto já é suficiente para a comparação didática.

**Interpretação dos resultados**

Os gráficos devem mostrar uma tendência linear para as três soluções.
A diferença real está nas constantes de tempo e na quantidade de etapas intermediárias:

- a solução 1 faz mais trabalho conceitual ao reconstruir números inteiros completos;
- a solução 2 remove uma parte dessa sobrecarga;
- a solução 3 trabalha diretamente sobre os nós e tende a ser a mais natural e eficiente.

Como as três abordagens são lineares, é comum que os tempos fiquem relativamente próximos para entradas pequenas.
Mesmo assim, a solução 3 costuma ser a que melhor reflete o raciocínio ideal do problema.

Em medições curtas, ruído de ambiente e variações do interpretador podem afetar a ordem exata dos resultados.
Por isso, o gráfico é um apoio didático, não uma prova formal.

**Comparação entre as abordagens**

| Solução | Ideia central | Tempo | Espaço | Vantagens | Desvantagens | Quando usar |
|---|---|---:|---:|---|---|---|
| Força bruta | Converter tudo para inteiro, somar e converter de volta | `O(m + n)` | `O(m + n)` | Muito fácil de entender | Faz conversões completas e não explora bem a estrutura do problema | Como primeiro passo de estudo |
| Melhorada | Trabalhar com listas auxiliares de dígitos e soma manual | `O(m + n)` | `O(m + n)` | Mais próxima do problema e mais clara que a solução 1 | Ainda usa estruturas intermediárias | Como evolução didática |
| Solução ótima | Percorrer os nós diretamente com carry | `O(max(m, n))` | `O(max(m, n))` para a saída | Mais direta, elegante e eficiente | Exige pensar de forma incremental | Em entrevistas e produção |

**Observação sobre a solução 3**

A solução 3 tem duas variantes no notebook: uma versão didática e uma versão enxuta. As duas continuam assintoticamente ótimas; a variante enxuta tenta apenas melhorar o custo constante e a concisão do código.

**Qual usar?**

- **Em entrevistas:** a solução ótima é a resposta esperada, e a versão didática ajuda a explicar o raciocínio.
- **Em produção:** a versão enxuta costuma ser a escolha mais prática, desde que o código continue legível.

Se o objetivo for aprender, vale entender as três versões. Se o objetivo for resolver bem, a terceira é a que realmente importa.

**Conclusão**

O problema **Add Two Numbers** ensina uma lição muito útil: às vezes, a melhor solução não é transformar o problema em outra coisa, mas sim trabalhar diretamente com a estrutura que ele já oferece.

A solução ótima é melhor porque:

- aproveita o fato de os dígitos estarem em ordem reversa;
- processa os nós um por um;
- lida com o `carry` naturalmente;
- evita conversões intermediárias desnecessárias.

Esse tipo de problema reforça conceitos importantes sobre listas encadeadas, fluxo incremental e construção de estruturas de saída.

**Possíveis variações**

- listas encadeadas com dígitos em ordem normal;
- soma de mais de dois números;
- implementação recursiva;
- problemas que pedem remoção, inversão ou junção de listas encadeadas.

Com isso, fechamos um capítulo bem completo sobre a resolução do problema e sobre como pensar em linked lists de forma prática.


In [ ]:
tamanhos = [1, 2, 5, 10, 20, 40, 80, 100]
curvas = pd.DataFrame(
    {
        'n': tamanhos,
        'forca_bruta': [3 * n for n in tamanhos],
        'melhorada': [2 * n for n in tamanhos],
        'otimizada': [1 * n for n in tamanhos],
    }
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=curvas, x='n', y='forca_bruta', marker='o', label='Solução 1', ax=ax)
sns.lineplot(data=curvas, x='n', y='melhorada', marker='o', label='Solução 2', ax=ax)
sns.lineplot(data=curvas, x='n', y='otimizada', marker='o', label='Solução 3', ax=ax)
ax.set_title('Crescimento didático dos custos relativos')
ax.set_xlabel('Tamanho da lista (n)')
ax.set_ylabel('Custo relativo conceitual')
plt.tight_layout()
plt.show()


In [ ]:
def gerar_caso(tamanho: int) -> tuple[list[int], list[int]]:
    """Gera um caso de teste com listas de dígitos reversos.

    O caso usa números formados por muitos 9s para forçar o carry em várias
    posições e tornar a comparação mais interessante didaticamente.
    """
    l1 = [9] * tamanho
    l2 = [9] * tamanho
    return l1, l2


def medir_solucao(func, tamanhos: list[int], repeticoes: int = 20) -> pd.DataFrame:
    registros = []
    for tamanho in tamanhos:
        casos = [gerar_caso(tamanho) for _ in range(repeticoes)]
        tempos: list[float] = []
        for l1_values, l2_values in casos:
            l1 = list_to_linked_list(l1_values)
            l2 = list_to_linked_list(l2_values)
            inicio = perf_counter()
            resultado = func(l1, l2)
            fim = perf_counter()
            assert isinstance(resultado, ListNode)
            tempos.append(fim - inicio)

        registros.append(
            {
                'tamanho': tamanho,
                'tempo_medio_s': float(np.mean(tempos)),
                'solucao': func.__name__,
            }
        )

    return pd.DataFrame(registros)


tamanhos_benchmark = [1, 2, 5, 10, 20, 40, 80, 100]
df_benchmark = pd.concat(
    [
        medir_solucao(add_two_numbers_brute_force, tamanhos_benchmark),
        medir_solucao(add_two_numbers_better, tamanhos_benchmark),
        medir_solucao(add_two_numbers_optimized, tamanhos_benchmark),
    ],
    ignore_index=True,
)

df_benchmark


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
sns.lineplot(
    data=df_benchmark,
    x='tamanho',
    y='tempo_medio_s',
    hue='solucao',
    marker='o',
    linewidth=2.5,
    ax=ax,
)
ax.set_title('Tempo médio observado por solução')
ax.set_xlabel('Tamanho da entrada (n)')
ax.set_ylabel('Tempo médio de execução (s)')
plt.tight_layout()
plt.show()
